# Lab 4: AgentCore Deployment (25 minutes)

In this lab, you will deploy your intelligent translation agent to Amazon Bedrock AgentCore Runtime for production use with full observability.

## Learning Objectives

**You'll Learn:**
- ✅ Package agent code for AgentCore deployment
- ✅ Deploy to AgentCore Runtime using CLI
- ✅ Test production endpoints
- ✅ Monitor agent execution with observability traces


## Step 1: Install AgentCore CLI

The AgentCore CLI simplifies deployment and management.

In [2]:
# Install AgentCore starter toolkit (includes CLI)
!pip install -q bedrock-agentcore-starter-toolkit


print("✅ AgentCore CLI installed successfully!")

✅ AgentCore CLI installed successfully!


## Step 2: Create Agent Entrypoint

AgentCore requires a specific entrypoint structure.

In [3]:
%%writefile lab_helpers/production_agent.py
"""
Production translation agent for AgentCore deployment.
"""

from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
import boto3
import os

# Initialize AgentCore app
app = BedrockAgentCoreApp()

# Get Knowledge Base ID from environment
KB_ID = os.environ.get('KNOWLEDGE_BASE_ID', '')

# Define tools
@tool
def evaluate_translation_quality(
    source_text: str,
    translated_text: str,
    target_language: str = "Russian"
) -> dict:
    """Evaluate translation quality and identify issues."""
    return {
        "quality_score": 0,
        "issues": [],
        "needs_refinement": True,
        "evaluation_prompt": f"""
        Evaluate this {target_language} translation:
        Source: {source_text}
        Translation: {translated_text}
        
        Check accuracy, fluency, terminology, and tone.
        Provide quality_score (0-100), issues list, and needs_refinement flag.
        """
    }

@tool
def query_aws_terminology(term: str) -> str:
    """Query AWS terminology Knowledge Base."""
    if not KB_ID:
        return f"Knowledge Base not configured for term: {term}"
    
    try:
        bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')
        query = f"What is the correct Russian translation for '{term}'?"
        
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {'numberOfResults': 1}
            }
        )
        
        if response.get('retrievalResults'):
            content = response['retrievalResults'][0].get('content', {}).get('text', '')
            return f"Terminology for '{term}': {content}"
        return f"No terminology found for '{term}'"
    except Exception as e:
        return f"Error querying terminology: {str(e)}"

@tool
def refine_translation_section(
    original_translation: str,
    section_to_refine: str,
    refinement_instructions: str
) -> str:
    """Refine specific section of translation."""
    return f"Refine: {section_to_refine} - Instructions: {refinement_instructions}"

# Configure model
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0"
)

# Create agent
agent = Agent(
    model=model,
    tools=[
        evaluate_translation_quality,
        query_aws_terminology,
        refine_translation_section
    ],
    system_prompt="""
    You are a production-quality translation agent.
    
    Workflow:
    1. Translate English to Russian
    2. Validate technical terms using query_aws_terminology
    3. Evaluate quality using evaluate_translation_quality
    4. Refine if quality_score < 85
    5. Repeat until quality >= 85
    
    Rules:
    - Keep AWS service names in English
    - Use natural, fluent Russian
    - Validate all technical terms
    - Show iteration count and validated terms
    """
)

@app.entrypoint
def invoke(payload, context):
    """AgentCore entrypoint for translation requests."""
    user_message = payload.get(
        "prompt",
        "Please provide text to translate in the 'prompt' field"
    )
    
    result = agent(user_message)
    
    return {
        "result": result.message,
        "iterations": len(agent.messages) // 2,
        "status": "complete"
    }

if __name__ == "__main__":
    app.run()

print("✅ Production agent entrypoint created!")

Writing lab_helpers/production_agent.py


## Step 3: Create Requirements File for Deployment

In [ ]:
%%writefile lab_helpers/requirements.txt
bedrock-agentcore
strands-agents
boto3

print("✅ Deployment requirements file created!")

## Step 4: Configure AgentCore Deployment

The `agentcore configure` command sets up deployment parameters.

In [4]:
import sys
sys.path.append('lab_helpers')
from utils import get_aws_region, print_section_header, print_success, print_info
import boto3

print_section_header("Configuring AgentCore Deployment")

# Get Knowledge Base ID from CloudFormation
cfn_client = boto3.client('cloudformation')
try:
    response = cfn_client.describe_stacks(StackName='translation-workshop-kb')
    outputs = response['Stacks'][0]['Outputs']
    kb_id = next(
        (o['OutputValue'] for o in outputs if o['OutputKey'] == 'KnowledgeBaseId'),
        None
    )
    print_info(f"Knowledge Base ID: {kb_id}")
except Exception as e:
    print(f"⚠️ Could not retrieve KB ID: {e}")
    kb_id = ""

print_info(f"Region: {get_aws_region()}")
print_info("Entrypoint: lab_helpers/production_agent.py")

print("\n📝 Run this command in terminal:")
print(f"""
cd lab_helpers
agentcore configure \\
  -e production_agent.py \\
  --env KNOWLEDGE_BASE_ID={kb_id}
""")


  Configuring AgentCore Deployment

⚠️ Could not retrieve KB ID: 'Outputs'
ℹ️  Region: us-east-1
ℹ️  Entrypoint: lab_helpers/production_agent.py

📝 Run this command in terminal:

cd lab_helpers
agentcore configure \
  -e production_agent.py \
  --env KNOWLEDGE_BASE_ID=



## Step 5: Deploy to AgentCore Runtime

**Note:** Run these commands in the terminal, not in the notebook.

In [ ]:
print_section_header("Deployment Commands")

print("""
🚀 Deploy your agent to AgentCore Runtime:

1. Open a terminal in the lab_helpers directory
2. Run the following commands:

   # Configure deployment (if not done in Step 4)
   agentcore configure -e production_agent.py --env KNOWLEDGE_BASE_ID=<your-kb-id>
   
   # Launch deployment
   agentcore launch
   
   This will:
   - Package your agent code
   - Create Docker container
   - Deploy to AgentCore Runtime
   - Provision serverless infrastructure
   
   ⏳ Deployment takes 5-10 minutes

3. Once complete, you'll receive:
   - Agent endpoint URL
   - Agent ARN
   - Invocation instructions
""")

print("\n💡 Tip: Keep the terminal output - you'll need the agent ARN for testing")

## Step 6: Test Production Endpoint

After deployment completes, test your agent.

In [ ]:
print_section_header("Testing Production Agent")

# TODO: Replace with your actual agent ARN from deployment
AGENT_ARN = "<your-agent-arn-here>"

print("""
🧪 Test your deployed agent:

Method 1 - Using AgentCore CLI:

   agentcore invoke '{"prompt": "Translate to Russian: AWS Lambda is a serverless compute service"}'

Method 2 - Using Python SDK:
""")

print("""
from bedrock_agentcore import BedrockAgentCoreClient

client = BedrockAgentCoreClient()

response = client.invoke_agent(
    agent_arn=AGENT_ARN,
    payload={
        "prompt": "Translate to Russian: AWS Lambda is a serverless compute service"
    }
)

print(response['result'])
""")

print("\n💡 Uncomment and run the code above after replacing AGENT_ARN")

## Step 7: Monitor with AgentCore Observability

View execution traces in CloudWatch.

In [ ]:
print_section_header("AgentCore Observability")

print("""
📊 Monitor your agent execution:

1. CloudWatch Logs:
   - Navigate to CloudWatch Console
   - Find log group: /aws/bedrock/agentcore/<agent-name>
   - View execution logs and traces

2. AgentCore Observability Dashboard:
   - View agent invocation metrics
   - See tool usage statistics
   - Monitor iteration counts
   - Track quality scores

3. X-Ray Traces:
   - Detailed execution timeline
   - Tool call latencies
   - Knowledge Base query performance
   - End-to-end request tracing

🔍 What to look for:
   - How many iterations did the agent perform?
   - Which tools were called and in what order?
   - How long did KB queries take?
   - What was the final quality score?
""")

print("\n💡 Observability helps you understand agent behavior in production")

## Step 8: Production Testing Scenarios

In [ ]:
print_section_header("Production Test Scenarios")

test_scenarios = [
    {
        "name": "Simple Technical Term",
        "prompt": "Translate to Russian: Lambda functions scale automatically"
    },
    {
        "name": "Complex Paragraph",
        "prompt": "Translate to Russian: AWS Lambda is a serverless compute service that runs your code in response to events and automatically manages the compute resources."
    },
    {
        "name": "Multiple Technical Terms",
        "prompt": "Translate to Russian: Lambda integrates with S3, DynamoDB, and API Gateway for event-driven architectures."
    }
]

print("🧪 Recommended test scenarios:\n")
for i, scenario in enumerate(test_scenarios, 1):
    print(f"{i}. {scenario['name']}")
    print(f"   Prompt: {scenario['prompt']}")
    print(f"   Command: agentcore invoke '{{\"prompt\": \"{scenario['prompt']}\"}}'")
    print()

print("\n📊 For each test, observe:")
print("   - Translation quality")
print("   - Terminology consistency")
print("   - Iteration count")
print("   - Execution time")

## Step 9: Cleanup (Optional)

Remove deployed resources when done.

In [ ]:
print_section_header("Cleanup Commands")

print("""
🧹 To remove deployed resources:

1. Delete AgentCore Runtime deployment:
   agentcore delete

2. Delete Knowledge Base CloudFormation stack:
   aws cloudformation delete-stack --stack-name translation-workshop-kb

3. Delete S3 bucket (after emptying it):
   aws s3 rb s3://<bucket-name> --force

⚠️ Warning: This will permanently delete all resources.
   Only run cleanup when you're completely done with the workshop.
""")

## Lab 4 Complete - Workshop Summary

### 🎉 What You've Accomplished:

✅ **Complete Workshop Journey:**
- Lab 1: Identified translation quality problems
- Lab 2: Built self-evaluating agent with iteration
- Lab 3: Integrated Knowledge Base for terminology
- Lab 4: Deployed to production with observability

✅ **Agentic AI Patterns Mastered:**
- Reflect-refine loops for autonomous improvement
- Self-evaluation and quality assessment
- External knowledge integration
- Multi-tool coordination
- Production deployment and monitoring

✅ **AWS Services Integrated:**
- Amazon Bedrock (Claude 3.7 Sonnet)
- Bedrock Knowledge Base
- AgentCore Runtime
- AgentCore Observability
- CloudFormation
- S3

### 🎯 Key Differentiators:

**vs. Traditional Translation:**
- Autonomous quality improvement
- Terminology consistency enforcement
- Self-correcting behavior

**vs. Bedrock Data Automation:**
- Conversational interaction
- Iterative refinement
- Context maintenance
- Tool-based decision making

### 💡 Real-World Applications:

This pattern applies beyond translation:
- Code review with iterative improvements
- Document summarization with quality checks
- Data analysis with validation loops
- Content generation with brand consistency

### 🚀 Next Steps:

1. **Extend the Agent:**
   - Add more language pairs
   - Integrate translation memory
   - Add formality level controls

2. **Enhance Quality:**
   - Add COMET scoring
   - Implement A/B testing
   - Add human-in-the-loop review

3. **Scale to Production:**
   - Add rate limiting
   - Implement caching
   - Set up CI/CD pipeline
   - Add cost monitoring

### 📚 Additional Resources:

- [Strands Agents Documentation](https://strandsagents.com)
- [AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/)
- [Bedrock Knowledge Bases](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html)

**🎓 Congratulations on completing the Intelligent Translation Agent workshop!**
